# 02. Dataset Analysis

This notebook analyzes the Fakeddit dataset structure and prepares it for training.

In [ ]:
import sys
from pathlib import Path

# Add project root to path
sys.path.insert(0, str(Path('.').absolute()))

import pandas as pd
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns

from src.dataset import MultimodalFakeNewsDataset
from src.utils import set_seed, ensure_dir

## Dataset Configuration

**IMPORTANT**: Before running this notebook, download the Fakeddit dataset from:
- Website: https://fakeddit.netlify.app/
- GitHub: https://github.com/entitize/Fakeddit

Update the paths below to match your local setup.

In [ ]:
# Dataset paths - UPDATE THESE FOR YOUR SETUP
FAKEDIT_CSV_PATH = "data/fakeddit/your_fakeddit_file.csv"  # Update this
FAKEDIT_IMAGES_DIR = "data/fakeddit/images"  # Update this

# Check if paths exist
print(f"CSV path exists: {Path(FAKEDIT_CSV_PATH).exists()}")
print(f"Images dir exists: {Path(FAKEDIT_IMAGES_DIR).exists()}")

## Loading and Exploring the Dataset

In [ ]:
# Load the CSV file
df = pd.read_csv(FAKEDIT_CSV_PATH)

print(f"Dataset shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nFirst few rows:")
df.head()

In [ ]:
# Check column data types
print("Data types:")
print(df.dtypes)

# Check for missing values
print("\nMissing values:")
print(df.isnull().sum())

## Analyzing Label Distribution

Fakeddit uses distant supervision labels. Understanding the label distribution is crucial.

In [ ]:
# Check label column (adjust column name based on actual data)
# Common label columns in Fakeddit: 'label', 'category', '2_way_label', '3_way_label'

label_columns = ['label', 'category', '2_way_label', '3_way_label']
available_label_col = None

for col in label_columns:
    if col in df.columns:
        available_label_col = col
        print(f"Found label column: {col}")
        break

if available_label_col:
    print(f"\nLabel distribution for '{available_label_col}':")
    print(df[available_label_col].value_counts())
    
    # Plot
    fig, ax = plt.subplots(figsize=(8, 4))
    df[available_label_col].value_counts().plot(kind='bar', ax=ax)
    ax.set_title(f'Label Distribution ({available_label_col})')
    ax.set_xlabel('Label')
    ax.set_ylabel('Count')
    plt.tight_layout()
    plt.show()
else:
    print("No standard label column found. Please check column names.")

## Text Analysis

In [ ]:
# Find text column
text_columns = ['clean_title', 'title', 'text', 'content']
text_col = None

for col in text_columns:
    if col in df.columns:
        text_col = col
        print(f"Found text column: {col}")
        break

if text_col:
    # Text length statistics
    df['text_length'] = df[text_col].astype(str).apply(len)
    df['word_count'] = df[text_col].astype(str).apply(lambda x: len(x.split()))
    
    print(f"\nText length statistics:")
    print(df['text_length'].describe())
    
    print(f"\nWord count statistics:")
    print(df['word_count'].describe())
    
    # Plot text length distribution
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    axes[0].hist(df['text_length'], bins=50)
    axes[0].set_title('Text Length Distribution')
    axes[0].set_xlabel('Characters')
    axes[0].set_ylabel('Count')
    
    axes[1].hist(df['word_count'], bins=50)
    axes[1].set_title('Word Count Distribution')
    axes[1].set_xlabel('Words')
    axes[1].set_ylabel('Count')
    
    plt.tight_layout()
    plt.show()

## Image Analysis

In [ ]:
# Find image column
image_columns = ['image_id', 'image', 'image_path', 'id']
image_col = None

for col in image_columns:
    if col in df.columns:
        image_col = col
        print(f"Found image column: {col}")
        break

print(f"\nTotal samples: {len(df)}")
print(f"Non-null images: {df[image_col].notna().sum()}")

In [ ]:
# Check image availability (sample a few)
if image_col:
    sample_images = df[image_col].dropna().head(10).tolist()
    available_count = 0
    
    for img_id in sample_images:
        img_path = Path(FAKEDIT_IMAGES_DIR) / f"{img_id}.jpg"
        if img_path.exists():
            available_count += 1
    
    print(f"Sample image availability: {available_count}/{len(sample_images)}")

## Creating Dataset Splits

In [ ]:
# Configuration for dataset
TEXT_COLUMN = text_col if text_col else 'clean_title'
IMAGE_COLUMN = image_col if image_col else 'image_id'
LABEL_COLUMN = available_label_col if available_label_col else 'label'

# Binary label mapping for fake news
# Fakeddit 3-way: 0=reliable, 1=unreliable, 2=misleading
# Binary: 0=real, 1=fake
LABEL_MAPPING = {
    0: 0,  # reliable -> real
    1: 1,  # unreliable -> fake
    2: 1   # misleading -> fake
}

print(f"Text column: {TEXT_COLUMN}")
print(f"Image column: {IMAGE_COLUMN}")
print(f"Label column: {LABEL_COLUMN}")
print(f"Label mapping: {LABEL_MAPPING}")

In [ ]:
# Apply label mapping
df['binary_label'] = df[LABEL_COLUMN].map(LABEL_MAPPING)

# Check mapping results
print("Binary label distribution:")
print(df['binary_label'].value_counts())

# Filter out unmapped labels
df_clean = df[df['binary_label'].notna()].copy()
print(f"\nCleaned dataset size: {len(df_clean)}")

## Save Processed Dataset

In [ ]:
# Save processed dataset info
output_dir = Path('data/processed')
ensure_dir(output_dir)

# Save a summary CSV
summary_df = df_clean[[TEXT_COLUMN, IMAGE_COLUMN, 'binary_label']].copy()
summary_df.to_csv(output_dir / 'fakeddit_binary.csv', index=False)

print(f"Saved processed dataset to {output_dir / 'fakeddit_binary.csv'}")

# Create and save a split manifest
from sklearn.model_selection import train_test_split

set_seed(42)

# Split
train_df, temp_df = train_test_split(
    summary_df, 
    test_size=0.3, 
    random_state=42, 
    stratify=summary_df['binary_label']
)

val_df, test_df = train_test_split(
    temp_df, 
    test_size=0.5, 
    random_state=42, 
    stratify=temp_df['binary_label']
)

print(f"\nDataset splits:")
print(f"  Train: {len(train_df)} ({len(train_df)/len(summary_df)*100:.1f}%)")
print(f"  Val: {len(val_df)} ({len(val_df)/len(summary_df)*100:.1f}%)")
print(f"  Test: {len(test_df)} ({len(test_df)/len(summary_df)*100:.1f}%)")

# Save splits
train_df.to_csv(output_dir / 'train.csv', index=False)
val_df.to_csv(output_dir / 'val.csv', index=False)
test_df.to_csv(output_dir / 'test.csv', index=False)

print(f"\nSaved split manifests to {output_dir}")

## Summary

In [ ]:
print("\n" + "="*50)
print("DATASET ANALYSIS COMPLETE")
print("="*50)
print(f"\nDataset: Fakeddit")
print(f"Total samples: {len(df_clean)}")
print(f"Real news: {(df_clean['binary_label'] == 0).sum()}")
print(f"Fake news: {(df_clean['binary_label'] == 1).sum()}")
print(f"\nNext steps:")
print("1. Run 03_baseline_models.ipynb to train text-only and image-only baselines")
print("2. Run 04_consistency_aware_training.ipynb to train the full model")